# Indic ASR Dataset Ecosystem — Pipeline Notebook

This notebook walks through the complete pipeline:

1. **Setup** — install dependencies, clone/mount the project
2. **Discovery** — find new Indic ASR datasets on HF Hub
3. **Ingest** — run adapters and write registry entries
4. **Validate** — check all registry entries for correctness
5. **Build Manifests** — generate model-ready Parquet files
6. **Inspect** — explore the manifests
7. **Publish** — push manifests to HuggingFace Hub

---
**Architecture overview:**
```
seed_datasets.yaml  →  adapters  →  registry/entries/*.json
                                            ↓
                              manifests/*.parquet  →  HF Hub
```

No audio is downloaded or stored — only pointers.

In [ ]:
# ============================================================
# CELL 1: Install dependencies
# ============================================================
!pip install -q \
    datasets \
    huggingface_hub \
    pydantic>=2.0 \
    pyyaml \
    pyarrow \
    pandas \
    tqdm

print('Dependencies installed.')

In [ ]:
# ============================================================
# CELL 2: Clone the project (or mount from Drive)
# ============================================================
import os
import sys

# Option A: Clone from GitHub
# !git clone https://github.com/YOUR_ORG/indic-asr-ecosystem /content/indic_asr_ecosystem

# Option B: Mount from Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/indic_asr_ecosystem'

# Option C: Use the project structure created alongside this notebook
PROJECT_ROOT = '/content/indic_asr_ecosystem'

# If running this notebook standalone, create the project structure
if not os.path.exists(PROJECT_ROOT):
    os.makedirs(f'{PROJECT_ROOT}/catalogue', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/registry/entries', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/manifests', exist_ok=True)
    print(f'Created project structure at {PROJECT_ROOT}')
    print('NOTE: Copy your seed_datasets.yaml and Python source files here.')
else:
    print(f'Using existing project at {PROJECT_ROOT}')

sys.path.insert(0, PROJECT_ROOT)
print(f'Python path: {PROJECT_ROOT}')

In [ ]:
# ============================================================
# CELL 3: Authenticate with HuggingFace Hub
#
# Required for:
#  - Accessing gated datasets (some Common Voice versions)
#  - Publishing manifests to your HF repo
#
# Get your token at: https://huggingface.co/settings/tokens
# ============================================================
from huggingface_hub import login
from google.colab import userdata

# Option A: Use Colab secrets (recommended)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('Logged in via Colab secret.')
except Exception:
    # Option B: Interactive login
    login()
    print('Logged in interactively.')

In [ ]:
# ============================================================
# CELL 4: [OPTIONAL] Discover new datasets
#
# This runs targeted searches on HF Hub to find Indic ASR datasets.
# Output is a candidate YAML for human review — NOT auto-added.
# ============================================================
import subprocess

result = subprocess.run(
    [
        'python', f'{PROJECT_ROOT}/scripts/discover.py',
        '--output', f'{PROJECT_ROOT}/discovered_candidates.yaml',
        '--orgs', 'ai4bharat', 'mozilla-foundation', 'google',
    ],
    capture_output=True, text=True
)
print(result.stdout[-3000:])  # Last 3000 chars
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# ============================================================
# CELL 5: Review discovered candidates
# ============================================================
candidates_path = f'{PROJECT_ROOT}/discovered_candidates.yaml'

if os.path.exists(candidates_path):
    with open(candidates_path) as f:
        content = f.read()
    # Show first 100 lines
    lines = content.split('\n')
    print(f'Total candidates: {content.count("- id:")}')
    print('\n'.join(lines[:100]))
else:
    print('No candidates file found. Run CELL 4 first.')

In [ ]:
# ============================================================
# CELL 6: Dry-run ingest (verify before writing)
#
# Always run --dry-run first to check for errors
# ============================================================

result = subprocess.run(
    [
        'python', f'{PROJECT_ROOT}/scripts/ingest.py',
        '--seed', f'{PROJECT_ROOT}/catalogue/seed_datasets.yaml',
        '--registry', f'{PROJECT_ROOT}/registry',
        '--dry-run',
        '--langs', 'hi', 'ta',  # Start with Hindi and Tamil
    ],
    capture_output=True, text=True,
    cwd=PROJECT_ROOT
)
print(result.stdout[-5000:])
if result.returncode != 0:
    print('STDERR:', result.stderr[-2000:])

In [ ]:
# ============================================================
# CELL 7: Run actual ingest
#
# This writes JSON files to registry/entries/
# Commit these files to git for versioning
# ============================================================

result = subprocess.run(
    [
        'python', f'{PROJECT_ROOT}/scripts/ingest.py',
        '--seed', f'{PROJECT_ROOT}/catalogue/seed_datasets.yaml',
        '--registry', f'{PROJECT_ROOT}/registry',
        '--langs', 'hi', 'ta',  # Adjust as needed
        # '--overwrite',        # Uncomment to refresh existing entries
    ],
    capture_output=True, text=True,
    cwd=PROJECT_ROOT
)
print(result.stdout)
if result.returncode != 0:
    print('ERRORS:', result.stderr[-2000:])

In [ ]:
# ============================================================
# CELL 8: Inspect registry entries
# ============================================================
import json
from pathlib import Path

entries_dir = Path(f'{PROJECT_ROOT}/registry/entries')
entries = sorted(entries_dir.glob('*.json'))
print(f'Registry entries: {len(entries)}')
print()

for entry_path in entries[:5]:  # Show first 5
    with open(entry_path) as f:
        data = json.load(f)
    print(f"{'='*60}")
    print(f"ID:       {data['entry_id']}")
    print(f"Language: {data['language']}")
    print(f"Split:    {data['split']}")
    print(f"Source:   {data.get('hf_repo_id')} / {data.get('hf_config')}")
    print(f"License:  {data['license']}")
    print(f"Quality:  {data['quality_tier']}")
    stats = data.get('stats')
    if stats:
        print(f"Samples:  {stats.get('num_samples', 'N/A')}")
    prov = data.get('provenance', {})
    print(f"Ingested: {prov.get('ingested_at')}")
    print(f"Commit:   {prov.get('hf_commit_sha', 'N/A')}")
    print()

In [ ]:
# ============================================================
# CELL 9: Validate all registry entries
# ============================================================
import sys
sys.path.insert(0, PROJECT_ROOT)

from indic_asr.registry.validator import validate_all_entries

validation_errors = validate_all_entries(f'{PROJECT_ROOT}/registry')

if not validation_errors:
    print('✓ All registry entries are valid.')
else:
    print(f'✗ {len(validation_errors)} entries have validation errors:')
    for entry_id, errors in validation_errors.items():
        print(f'\n  {entry_id}:')
        for err in errors:
            print(f'    - {err}')

In [ ]:
# ============================================================
# CELL 10: Build manifests
#
# This streams through source datasets to build Parquet manifests.
# For large datasets, this can take a while.
# The manifests are model-ready and deduplicated.
# ============================================================
from indic_asr.manifest.generator import ManifestGenerator, ManifestFilters
from indic_asr.schema import LicenseType, QualityTier

# Configure quality filters
# For training: be permissive
# For benchmarking: be strict
filters = ManifestFilters(
    min_duration_s=0.5,
    max_duration_s=20.0,
    min_transcript_chars=2,
    max_transcript_chars=300,
    # Uncomment to restrict licenses:
    # allowed_licenses=[LicenseType.CC0, LicenseType.CC_BY, LicenseType.CC_BY_SA],
    # Uncomment to restrict quality:
    # allowed_quality_tiers=[QualityTier.GOLD, QualityTier.SILVER],
    exclude_synthetic=False,
)

generator = ManifestGenerator(
    registry_dir=Path(f'{PROJECT_ROOT}/registry'),
    output_dir=Path(f'{PROJECT_ROOT}/manifests'),
    filters=filters,
    deduplicate=True,
)

# Build manifests for specific languages
results = generator.build_all_manifests(
    languages=['hi', 'ta'],  # Adjust as needed
)

print(f'\nBuilt {len(results)} manifests:')
for key, path in results.items():
    print(f'  {key}: {path}')

In [ ]:
# ============================================================
# CELL 11: Inspect manifests
# ============================================================
import pandas as pd
from pathlib import Path

manifests_dir = Path(f'{PROJECT_ROOT}/manifests')
parquet_files = sorted(manifests_dir.glob('*.parquet'))

print(f'Generated {len(parquet_files)} manifest files:\n')

for pf in parquet_files:
    df = pd.read_parquet(pf)
    print(f'{pf.name}:')
    print(f'  Rows: {len(df)}')
    print(f'  Languages: {df["language"].unique().tolist()}')
    print(f'  Sources: {df["source_entry_id"].nunique()} registry entries')
    
    if 'duration_seconds' in df.columns and df['duration_seconds'].notna().any():
        total_hrs = df['duration_seconds'].sum() / 3600
        print(f'  Total audio: {total_hrs:.1f} hours')
    
    if 'quality_tier' in df.columns:
        print(f'  Quality distribution:')
        for tier, count in df['quality_tier'].value_counts().items():
            print(f'    {tier}: {count}')
    
    if 'license' in df.columns:
        print(f'  License distribution:')
        for lic, count in df['license'].value_counts().items():
            print(f'    {lic}: {count}')
    
    print()
    print('  Sample rows:')
    print(df[['utterance_id', 'language', 'transcript', 'source_entry_id']].head(3).to_string())
    print()

In [ ]:
# ============================================================
# CELL 12: How to use the manifest in a training pipeline
#
# This demonstrates how Whisper, NeMo, or wav2vec2 training
# pipelines would consume the manifests.
# ============================================================

# Example: Load Hindi train manifest
import pandas as pd
from datasets import load_dataset, Audio

manifest_path = f'{PROJECT_ROOT}/manifests/hi_train.parquet'

if os.path.exists(manifest_path):
    df = pd.read_parquet(manifest_path)
    print(f'Loaded {len(df)} training examples for Hindi')

    # --- Pattern 1: Direct Parquet loading (simplest) ---
    # For frameworks that accept Parquet with audio pointers (NeMo format)
    # Just pass the manifest path to the framework

    # --- Pattern 2: Streaming from HF Hub ---
    # For each unique source in the manifest:
    for source_id in df['source_entry_id'].unique()[:2]:  # First 2 sources
        source_rows = df[df['source_entry_id'] == source_id]
        first_row = source_rows.iloc[0]
        
        print(f'\nSource: {source_id}')
        print(f'  HF repo: {first_row["audio_hf_repo"]}')
        print(f'  Config: {first_row["audio_hf_config"]}')
        print(f'  Split: {first_row["audio_hf_split"]}')
        print(f'  Example utterance: {first_row["utterance_id"]}')
        print(f'  Row index: {first_row["audio_hf_row_index"]}')
        
        # To fetch audio for a specific row:
        # ds = load_dataset(
        #     first_row['audio_hf_repo'],
        #     first_row['audio_hf_config'],
        #     split=first_row['audio_hf_split'],
        #     streaming=True,
        # ).cast_column('audio', Audio(sampling_rate=16000))
        # sample = next(itertools.islice(ds, first_row['audio_hf_row_index'], None))
        # audio_array = sample['audio']['array']
        # transcript = first_row['transcript']
else:
    print('No Hindi manifest found. Run CELL 10 first.')

In [ ]:
# ============================================================
# CELL 13: [OPTIONAL] Publish manifests to HuggingFace Hub
#
# Creates a HF dataset that points to the manifests.
# The manifests are uploaded; the audio stays in source repos.
# ============================================================
from huggingface_hub import HfApi, create_repo

HF_ORG = "your-org"  # Replace with your HF org/username
REPO_NAME = "indic-asr-manifests"
REPO_ID = f"{HF_ORG}/{REPO_NAME}"

# Create repo if it doesn't exist
try:
    create_repo(REPO_ID, repo_type="dataset", exist_ok=True)
    print(f'Repository ready: {REPO_ID}')
except Exception as e:
    print(f'Error creating repo: {e}')

# Upload manifests
api = HfApi()

manifests_dir = Path(f'{PROJECT_ROOT}/manifests')
for parquet_file in manifests_dir.glob('*.parquet'):
    api.upload_file(
        path_or_fileobj=str(parquet_file),
        path_in_repo=f'data/{parquet_file.name}',
        repo_id=REPO_ID,
        repo_type='dataset',
    )
    print(f'Uploaded: {parquet_file.name}')

# Upload metadata files
for meta_file in manifests_dir.glob('*_metadata.json'):
    api.upload_file(
        path_or_fileobj=str(meta_file),
        path_in_repo=f'metadata/{meta_file.name}',
        repo_id=REPO_ID,
        repo_type='dataset',
    )

# Upload registry (for full provenance)
api.upload_folder(
    folder_path=f'{PROJECT_ROOT}/registry',
    path_in_repo='registry',
    repo_id=REPO_ID,
    repo_type='dataset',
)

print(f'\nPublished to: https://huggingface.co/datasets/{REPO_ID}')

In [ ]:
# ============================================================
# CELL 14: Adapter development helper
#
# Use this to inspect a new dataset's schema before writing
# a custom adapter for it.
# ============================================================
from datasets import load_dataset_builder

def inspect_dataset(repo_id: str, config: str = None):
    """Inspect a HF dataset's schema without downloading data."""
    print(f'Inspecting: {repo_id} / {config}')
    print('=' * 60)
    
    builder = load_dataset_builder(repo_id, config, trust_remote_code=False)
    
    print(f'Name: {builder.info.builder_name}')
    print(f'Description: {builder.info.description[:200] if builder.info.description else "N/A"}')
    print(f'License: {builder.info.license}')
    print(f'Homepage: {builder.info.homepage}')
    print()
    
    if builder.info.features:
        print('Features (columns):')
        for col, dtype in builder.info.features.items():
            print(f'  {col}: {dtype}')
    print()
    
    if builder.info.splits:
        print('Splits:')
        for split_name, split_info in builder.info.splits.items():
            num = split_info.num_examples if split_info else 'N/A'
            print(f'  {split_name}: {num:,} examples' if isinstance(num, int) else f'  {split_name}: {num}')
    print()

# Examples:
inspect_dataset('mozilla-foundation/common_voice_17_0', 'hi')
# inspect_dataset('google/fleurs', 'hi_in')
# inspect_dataset('ai4bharat/shrutilipi')